> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验七：基于CANN的RoPE优化版算子实验


建议学时：4学时


# 实验任务


## 任务描述


本实验围绕Qwen2.5的RoPE计算构建AscendC自定义算子。算子数学语义为：。实验基于基础版算子进行开发，主要通过Profiling进行性能瓶颈分析、选择恰当性能优化方法、实际应用并逐一跑通数学正确性验证与Qwen模型前向传播验证，并记录各个优化方法的加速比。


## 学习目标


完成本任务的学习后，你应当能够使用 msprof 定位 RoPE 的 GM 标量访问瓶颈，理解动态核数、UB 分块 DataCopy、动态 tile 与 Mul/Sub/Add 向量化的实现；同时能够区分历史 compact/QK 融合探索与当前 CannLab 稳定实现，并以连续多次正确性测试和重新 benchmark 的结果评价优化收益。


# 任务准备


## 优化前的瓶颈与策略总览


优化动机：基础版为了便于验证，采用逐元素标量访问完成rotate_half计算。cos/sin按头维度展开存储（形状 [batch×numHeads×seqLen, headDim]），相同位置的三角函数表值被复制了多次，造成H2D传输冗余。每层Attention对Q和K分别独立调用一次算子，cos/sin被传输两次、stream同步执行两次。Host wrapper每次调用均走完整的CPU→NPU→CPU数据搬移路径（H2D + kernel + D2H），memcpy+sync占总耗时约85%。基础版的目的不是追求速度，而是让rotate_half公式与标量代码逐行对应，便于检查数值是否正确。优化版不改变RoPE的数学结果，而是在保持同一输入输出语义的前提下，从数据布局、任务划分、计算方式和调用路径四个层面系统性地消除冗余。


## 基础版与优化版的前后对比


当前稳定优化版不改变公式和对外接口语义，主要执行路径为“最多 32 核动态分工 → 按 UB 容量计算 tile → x/cos/sin 批量 DataCopy 到 UB → Mul/Sub/Add 向量计算 → CopyOut”。compact API 仍接收 [B,S,D] trig，但 wrapper 会先展开为完整二维表；Q/K 组合 API 在内部顺序执行两条稳定 tiled 路径。Kernel 内 compact 映射、真正 Q/K 融合和零拷贝 NPU-resident 仅作为历史探索记录。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">优化策略</th>
<th style="text-align:left;">基础版的实现</th>
<th style="text-align:left;">优化版的修改位置</th>
<th style="text-align:left;">改动带来的作用</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">紧凑cos/sin布局</td>
<td style="text-align:left;">cos/sin按head维度展开，同一seqPos的trig值为每个头复制一次</td>
<td style="text-align:left;">对外仍接收 [B,S,D] compact trig；当前 wrapper 展开为 [B,H,S,D] 后进入稳定二维 tiled kernel。</td>
<td style="text-align:left;">保持接口兼容并确保 910B3 多次运行结果稳定；当前不宣称 compact H2D 减少。</td>
</tr>
<tr>
<td style="text-align:left;">动态核数Tiling增量索引</td>
<td style="text-align:left;">coreNum固定为min(8, totalTokens)，每行row直接作为trig索引</td>
<td style="text-align:left;">核函数入口从startRow一次性推导起点的(batch, rowInBatch, seqPos)，行循环中以增量递推自动处理跨batch/跨head边界</td>
<td style="text-align:left;">短序列避免核调度开销超过计算收益；行切分不要求对齐batch边界，任意位置切分均可正确递推</td>
</tr>
<tr>
<td style="text-align:left;">向量化Tile路径</td>
<td style="text-align:left;">每行headDim/2次迭代中执行6次标量读取x/cos/sin、2次标量写回output</td>
<td style="text-align:left;">TPipe/TQue/TBuf管理三段 异步流水线；DataCopy将 数据整行搬入UB；Vector API在进行向量化的RoPE计算；DataCopy写回output</td>
<td style="text-align:left;">消除逐元素标量GM访问，每tile仅3次DataCopy（x/cos/sin）和1次写回；向量单元替代标量单元，AI Core利用率大幅提升</td>
</tr>
<tr>
<td style="text-align:left;">Q/K融合调用</td>
<td style="text-align:left;">Attention层对Q和K分别调用，各经历独立H2D、launch、sync、D2H</td>
<td style="text-align:left;">保留 rope_qk_compact 组合 API；内部顺序调用 Q、K 两条 rope_compact_npu 稳定路径。</td>
<td style="text-align:left;">统一调用入口；当前不宣称单次 cos/sin H2D、单次 sync 或真正 Q/K 融合。</td>
</tr>
</tbody></table>


## 算子定义与接口约定


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">当前工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">input、cos、sin、output均为float32</td>
</tr>
<tr>
<td style="text-align:left;">输入布局</td>
<td style="text-align:left;">对外支持展开布局和 [B,S,D] compact 输入；compact 输入在 wrapper 中展开后再传入 kernel。</td>
</tr>
<tr>
<td style="text-align:left;">行数</td>
<td style="text-align:left;">totalTokens = batch × numHeads × seqLen 紧凑模式下trigTokens = batch × seqLen</td>
</tr>
<tr>
<td style="text-align:left;">并行划分</td>
<td style="text-align:left;">coreNum = totalTokens ≤ 16 ? 8 : 32 rowsPerCore = (totalTokens + coreNum - 1) / coreNum</td>
</tr>
<tr>
<td style="text-align:left;">标准测试形状</td>
<td style="text-align:left;">totalTokens=128, headDim=64, coreNum=8, tileSize=8</td>
</tr>
</tbody></table>


## 实验环境准备


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">硬件</td>
<td style="text-align:left;">Ascend 910B4宿主NPU</td>
</tr>
<tr>
<td style="text-align:left;">工具链</td>
<td style="text-align:left;">CANN 8.5.0</td>
</tr>
<tr>
<td style="text-align:left;">框架接口</td>
<td style="text-align:left;">PyTorch C++ extension qwen_rope_custom_opt：rope_baseline(x,cos,sin)、rope_compact(x,cos,sin,seq_len,num_heads)、rope_qk_compact(q,k,cos,sin,seq_len,q_heads,k_heads)</td>
</tr>
<tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">FP32</td>
</tr>
<tr>
<td style="text-align:left;">构建产物</td>
<td style="text-align:left;">out/lib/libascendc_kernels_npu.so、librope_torch_register.so；out/bin/rope_baseline_standalone</td>
</tr>
<tr>
<td style="text-align:left;">测试参考</td>
<td style="text-align:left;">NumPy float64精度参考数据 可执行测试程序C++逐元素同公式</td>
</tr>
</tbody></table>


# 任务实施


## 步骤一：Profiling采集、瓶颈定位和优化方法选择


在基础版工程目录下运行独立可执行测试程序和msprof，采集AI Core性能指标。关键观察项：


aclrtMemcpy（H2D/D2H）占比较高时，应分别统计 wrapper 与 kernel 时间；当前版本仍包含 H2D/kernel/D2H，真正 NPU-resident 数据流属于后续优化方向。


aiv_scalar_time与aiv_vec_time的比值，若scalar_time占主导，说明标量GM访问是kernel内部瓶颈，应当引入恰当的Tiling并实现批量搬运与向量化计算；


Memory有效带宽，对比HBM标准值，基础版预期远低于峰值，实施DMA传输后应当能够显著提升。


根据msprof结论制定优化策略，按收益从高到低排序。实测数据表明memcpy+sync占总耗时约85%，kernel计算仅约29%，因此优先消除数据搬移，其次优化kernel内部数据流。


## 步骤二：compact API 的稳定实现


cos/sin 在数学上只依赖 batch 与序列位置，因此对外保留 [B,S,D] compact API。历史版本曾在 kernel 内按 trigRow 直接复用紧凑表，但该路径在 Ascend 910B3 上出现非确定性结果。当前稳定版改为 wrapper 执行 reshape→expand→contiguous→reshape，得到与输入逐行对应的完整 trig，再复用稳定二维 tiled kernel。


Tiling 结构仍保留 seqLen、numHeads、trigTokens、compactTrig 和 tileSize 字段，用于接口兼容、shape 校验和历史分支记录；当前稳定 compact 调用在进入 kernel 前已经展开 trig，并以普通二维 tiled 路径执行。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">#pragma pack(push, 1) struct RoPeTiling { uint32_t totalTokens = 0; uint32_t headDim = 0; uint32_t coreNum = 1; uint32_t rowsPerCore = 0; uint32_t seqLen = 0; uint32_t numHeads = 1; uint32_t trigTokens = 0; uint32_t compactTrig = 0; uint32_t tileSize = 0; // 0/1=标量, &gt;1=DataCopy+Vector }; #pragma pack(pop)</th>
</tr>
</thead>
</table>


历史 kernel 内 trig-row 映射逻辑保留在源码中作为实验记录，但当前稳定调用不进入该分支。


## 步骤三：动态Tiling与增量索引


当前生效的 kernel 优化包括动态核数和 tiled 向量路径：


动态核数选择：根据totalTokens决定blockDim。少量token（≤16）封顶8核，避免核调度开销超过并行收益；大批量token放开至32核。Host端wrapper中实现核心数选择逻辑：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">uint32_t coreNum = choose_core_num(totalTokens); uint32_t tileSize = compute_tile_size(headDim); auto cos_full = cos.reshape({batch, 1, seq_len, headDim}) .expand({batch, num_heads, seq_len, headDim}) .contiguous().reshape({totalTokens, headDim}); auto sin_full = sin.reshape({batch, 1, seq_len, headDim}) .expand({batch, num_heads, seq_len, headDim}) .contiguous().reshape({totalTokens, headDim}); return rope_launch(x, cos_full, sin_full, 0, 1, false); // rope_qk_compact 当前内部顺序调用： auto q_out = rope_compact_npu(q, cos, sin, seq_len, q_heads); auto k_out = rope_compact_npu(k, cos, sin, seq_len, k_heads);</th>
</tr>
</thead>
</table>


增量机制自动处理跨batch、跨head（即省去了将核心任务划分对齐词向量与注意力头的复杂计算）的情况，无论行切分在哪个位置都能正确递推。


向量化 Tile 路径：tileSize > 1 时，kernel 使用 TPipe/TQue/TBuf 管理局部数据，通过 DataCopy 将 x/cos/sin 从 GM 批量搬入 UB，使用 Mul/Sub/Add 完成 RoPE，最后批量 CopyOut。当前 compact API 已在 wrapper 展开 trig，因此进入普通二维 tiled 分支，不再依赖 VECCALC 中的 compact 标量展开。


## 步骤四：Q/K 组合接口与当前顺序执行


基础版每层Attention对Q和K分别调用RoPE算子，两个矩阵各经历独立的H2D(cos/sin)、launch、sync、D2H完整路径。两次传输相同的三角函数表，两次流同步。


优化版保留 rope_qk_compact(q, k, cos, sin, seq_len, q_heads, k_heads) 组合接口，但当前实现内部依次调用两次 rope_compact_npu。Q、K 分别展开 trig、启动 kernel 并同步，因此不能描述为真正融合、单次 trig H2D 或单次 sync。


Host wrapper内部：一次性拷贝q、k、cos、sin和Tiling数据；在同一ACL stream上先后启动 Q 计算和K 计算；仅进行一次流同步；一次性D2H回读旋转后的Q和K。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /YOURPATH/RopeOptimizedExperiment bash scripts/check_env.sh bash scripts/build.sh bash scripts/run_test.sh tests/test_torch_op.py # 单算子正确性 bash scripts/run_test.sh tests/test_qwen_forward.py # Qwen链路替换验bash scripts/run_test.sh tests/test_mini_qwen.py # 最小Qwen模型端到端 bash scripts/run_test.sh tests/test_npu_e2e.py # NPU-Resident端到端 TOTAL_TOKENS=”The capital of France is” HEAD_DIM=64 BLOCK_DIM=8 WARMUP=5 REPEAT=10 bash scripts/profile.sh</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RopeOptimizedExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RopeOptimizedExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
mkdir -p input output
python3 -c "import numpy as np; t,d=128,64; r=np.random.default_rng(42); x=r.normal(size=(t,d)).astype(np.float32); f=np.outer(np.arange(t,dtype=np.float64),1.0/(1000000.0**(np.arange(0,d,2,dtype=np.float64)/d))); e=np.concatenate([f,f],axis=-1).astype(np.float32); x.tofile('input/input_x.bin'); np.cos(e).tofile('input/input_cos.bin'); np.sin(e).tofile('input/input_sin.bin')"
./out/bin/rope_baseline_standalone --tokens 128 --head-dim 64 --block-dim 32 --warmup 20 --repeat 100 --rounds 5


# 任务拓展


## 逐项加速比分解


针对当前生效的动态核数、动态 tile、GM↔UB DataCopy 和向量化计算，在相同 shape 与计时口径下重新测试基础版和优化版。历史 compact、Q/K 融合及 NPU-resident 数据只能作为实验演进参考，不能作为当前稳定版的加速比。


## 形状泛化性验证


改变tokens（10 / 70 / 128 / 256 / 1792）、headDim（64 / 128）和blockDim（4 / 8 / 16 / 32），分别运行基础版与优化版独立可执行验证程序，验证优化手段在不同输入规模下的有效性。特别关注短序列（tokens≤16）是否因动态Tiling启发式避免了性能退化。


## msprof深度分析


在优化后的典型形状下重新采集msprof数据，对比基础版的关键指标变化——memcpy占比应显著下降、aiv_scalar_time仍占主导、有效带宽应有提升。将基础版与优化版的msprof关键指标填入对照表，形成可量化的优化闭环证据链。


## E2E模型级验证


使用当前集成测试比较原生 RoPE 与自定义 RoPE 的输出和端到端时间。计时必须明确包含 wrapper 的 H2D/kernel/D2H；真正零拷贝 NPU-resident 路径尚未实现。


# 实验总结


本实验在RoPE基础版的基础上，完成了profiling驱动的逐项优化闭环。


当前稳定版保留动态多核、UB 分块 DataCopy、动态 tile 与 Mul/Sub/Add 向量化，并通过 wrapper 展开 trig、Q/K 顺序调用解决 910B3 非确定性问题。历史 compact 映射、Q/K 融合与 NPU-resident 方案继续作为优化演进记录，性能结论需按当前代码重测。


优化版的核心价值在于"数据驱动"，优化方向不由直觉决定，而是由msprof采集的aclrtMemcpy占比、aiv_scalar_time与aiv_vec_time比值、Memory有效带宽等硬指标驱动；每一项优化对应六类策略中的明确手段，策略与瓶颈之间存在清晰的映射关系。


本实验建立了算子优化的标准流程：


profiling定位瓶颈 → 策略映射排序 → 逐项实施 → Golden验证 → 基准对比 → 迭代决策。
